# Guided knee R=8: method comparison

Side-by-side reconstructions per network (one column each), then their residuals, on a few
slices. Reads the per-volume dumps `scripts/dump_eval.py` writes
(`<run>/eval_dump/<volume>.h5`, made by `bigpurple/guided_recon_dump.sbatch`) -- no
networks are rebuilt for runs that have a dump.

A run with **no dump yet** (e.g. the I2SB runs) is computed here on just the chosen slices
(`COMPUTE_MISSING`), through the same code the dump uses (`dump_eval.dump_run` ->
`evaluation/tasks.py`), into `notebooks/_compare_cache/`. That needs torch and, for I2SB's
sampler, a GPU (run this notebook on bigpurple). The measurement is noise-free (val sigma 0),
so those slices see exactly what the dumped runs saw; only I2SB's bridge noise differs from
what a full dump of the same run would draw.

Run top to bottom. Change `METHODS`, `N_SAMPLES` or pin `SAMPLES` in the first cell.

In [ ]:
import os, sys, random
import numpy as np
import h5py
import matplotlib.pyplot as plt

REPO = (os.path.abspath(os.path.join(os.getcwd(), ".."))
        if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, REPO)
os.chdir(REPO)       # run configs hold repo-relative paths (split CSVs, smaps, ...)
RUNS_ROOT = os.environ.get("COMPARE_RUNS_ROOT", os.path.join(
    REPO, "trained_nets", "guided_recon", "knee", "s4_noise-vlow"))

# (column label, run directory name, dataset in its dump). "recon_nodc": I2SB's plain sampler
# (no CG data consistency) on the same bridge noise.
METHODS = [
    ("LPDSNet",       "lpdsnet_R8",                    "recon"),
    ("TGVN",          "tgvn_c6_R8_g1",                 "recon"),
    ("LGGS",          "lggs_R8_g1",                    "recon"),
    ("LGGS guide-W",  "lggs_guide_w_R8_g1",            "recon"),
    ("I2SB",          "sbunet_small_R8_g1_tau0.19",    "recon"),
    ("I2SB + guide",  "sbunet_small_x1_R8_g1_tau0.19", "recon"),
    # ("I2SB, no CG",         "sbunet_small_R8_g1_tau0.19",    "recon_nodc"),
    # ("I2SB + guide, no CG", "sbunet_small_x1_R8_g1_tau0.19", "recon_nodc"),
]
N_SAMPLES = 4
SAMPLES = None            # or pin them: [("file1000071", 15), ...]  (volume, source slice)
SAMPLE_SEED = 0           # which slices, when SAMPLES is None
COMPUTE_MISSING = True    # runs with no dump: compute the chosen slices here (torch + GPU)
CKPT = "net_bestval.ckpt" # the dumps' checkpoint (guided_recon_dump.sbatch default)
SEED = 1234               # the dumps' seed (guided_recon_dump.sbatch default)
FLIP = True               # the stored knee slices are upside down
CROP = True               # crop to the anatomy (drops the readout oversampling)
RESID_GAIN = 4.0          # residuals are shown x this
CACHE = os.environ.get("COMPARE_CACHE", os.path.join(REPO, "notebooks", "_compare_cache"))
print("runs under", RUNS_ROOT)

In [ ]:
def _dirs(run):
    """Where a run's volumes may be: its dump, then this notebook's cache."""
    return [d for d in (os.path.join(RUNS_ROOT, run, "eval_dump"),
                        os.path.join(CACHE, run, "eval_dump")) if os.path.isdir(d)]


def find(run, vol, sl):
    """(file, row) holding source slice `sl` of `vol` for `run`, or None."""
    for d in _dirs(run):
        p = os.path.join(d, vol + ".h5")
        if os.path.exists(p):
            with h5py.File(p, "r") as f:
                idx = [int(s) for s in f["slice_index"][()]]
            if sl in idx:
                return p, idx.index(sl)
    return None


def read(run, vol, sl, ds="recon"):
    """One slice of one run: the image `ds`, the shared images, its stored metrics, attrs."""
    p, i = find(run, vol, sl)
    with h5py.File(p, "r") as f:
        out = {k: f[k][i] for k in ("reference", "zero_filled", "guide", "organ_mask") if k in f}
        out["img"] = f[ds][i]
        suffix = "" if ds == "recon" else "_" + ds[len("recon_"):]
        out["metrics"] = {m: float(f[f"{m}{suffix}_slice"][i])
                          for m in ("psnr", "ssim", "nrmse") if f"{m}{suffix}_slice" in f}
        out["attrs"] = {k: f.attrs[k] for k in ("sigma", "seed", "R", "ckpt", "task")
                        if k in f.attrs}
    return out


def slice_pairs(run):
    """Every (volume, source slice) a run has."""
    pairs = set()
    for d in _dirs(run):
        for fn in os.listdir(d):
            if fn.endswith(".h5"):
                with h5py.File(os.path.join(d, fn), "r") as f:
                    pairs |= {(fn[:-3], int(s)) for s in f["slice_index"][()]}
    return pairs


runs = list(dict.fromkeys(r for _, r, _ in METHODS))
pairs = {r: slice_pairs(r) for r in runs}
dumped = [r for r in runs if pairs[r]]
print("have images :", dumped)
print("no dump yet :", [r for r in runs if r not in dumped])
if not dumped:
    raise SystemExit("no run has a dump under RUNS_ROOT: run bigpurple/guided_recon_dump.sbatch")

if SAMPLES is None:
    common = set.intersection(*(pairs[r] for r in dumped))
    rng = random.Random(SAMPLE_SEED)
    vols = sorted({v for v, _ in common})
    rng.shuffle(vols)                       # one slice per volume, spread over patients
    SAMPLES = [(v, rng.choice(sorted(s for vv, s in common if vv == v)))
               for v in vols[:N_SAMPLES]]
print("samples     :", SAMPLES)

In [ ]:
# Runs without these slices: compute them here, through dump_eval's own code path.
todo = [r for r in runs if not all(find(r, v, s) for v, s in SAMPLES)]
for r in [r for r in todo if not os.path.exists(os.path.join(RUNS_ROOT, r, CKPT))]:
    print(f"[skip] {r}: no dump and no {CKPT} in {os.path.join(RUNS_ROOT, r)}")
    todo.remove(r)
if todo and COMPUTE_MISSING:
    import argparse
    import torch
    sys.path.insert(0, os.path.join(REPO, "scripts"))
    import dump_eval as de
    from evaluation.metrics import build_metrics
    from evaluation.tasks import EVAL_KSPACE_TYPE, default_sigma, force_organ_mask

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    metrics = build_metrics(["psnr", "ssim", "nrmse"])
    keep = {}
    for v, s in SAMPLES:
        keep.setdefault(v, set()).add(s)
    lo, hi = min(s for _, s in SAMPLES), max(s for _, s in SAMPLES) + 1
    for run in todo:
        run_dir = os.path.join(RUNS_ROOT, run)
        cfg, model, n_params, step = de.load_run(run_dir, device, CKPT)
        force_organ_mask(cfg)                                   # as dump_eval's default
        cfg.setdefault("training", {})["val_kspace_type"] = EVAL_KSPACE_TYPE
        if cfg["task"] == "i2sb_recon":                         # the run's own sampler, + no-CG
            cfg.setdefault("eval", {}).update(also_nodc=True, inversion=False)
        args = argparse.Namespace(slices=f"{lo}:{hi}", workers=2, seed=SEED, limit=None,
                                  ckpt=CKPT, keep_slices=keep)
        n = de.dump_run(run_dir, cfg, model, n_params, step, args, sorted(keep), metrics,
                        default_sigma(cfg), os.path.join(CACHE, run, "eval_dump"))
        print(f"{run}: computed {n} slices ({cfg['task']})")
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
elif todo:
    print("not computed (COMPUTE_MISSING is off):", todo)

shown = [(lab, r, ds) for lab, r, ds in METHODS if all(find(r, v, s) for v, s in SAMPLES)]
print("columns:", [lab for lab, _, _ in shown])

In [ ]:
def prep(a):
    a = np.abs(a)
    return np.flipud(a) if FLIP else a


def crop_box(mask, pad=6):
    """Row/column slice of the anatomy's bounding box (+ pad), for CROP."""
    if not CROP or mask is None or not mask.any():
        return slice(None), slice(None)
    rows, cols = np.where(mask)
    return (slice(max(rows.min() - pad, 0), rows.max() + pad + 1),
            slice(max(cols.min() - pad, 0), cols.max() + pad + 1))


def window(ref, mask):
    """Display ceiling shared by every column of a row: the viewer's (99.9th pct x 0.85)."""
    v = ref[mask] if mask is not None and mask.any() else ref
    return float(np.percentile(v, 99.9)) * 0.85


# every slice once: the shared images from the first shown run, each column's own image
base = shown[0][1]
rows = []
for v, s in SAMPLES:
    b = read(base, v, s)
    mask = prep(b["organ_mask"]).astype(bool) if "organ_mask" in b else None
    row = dict(name=f"{v} / {s}", ref=prep(b["reference"]), mask=mask,
               zf=prep(b["zero_filled"]) if "zero_filled" in b else None,
               guide=prep(b["guide"]) if "guide" in b else None, cols=[])
    for lab, r, ds in shown:
        x = read(r, v, s, ds)
        row["cols"].append((lab, prep(x["img"]), x["metrics"], x["attrs"]))
    rows.append(row)

# columns must be the same problem: warn if a run was dumped at another sigma / seed / R
for row in rows:
    a0 = row["cols"][0][3]
    for lab, _, _, a in row["cols"][1:]:
        bad = [k for k in ("sigma", "seed", "R") if k in a0 and k in a and a0[k] != a[k]]
        if bad:
            print(f"[warn] {lab} differs from {row['cols'][0][0]} in {bad}")
    break

## Reconstructions
Each row one slice, each column one method; one display window per row (the reference's). Under each method: PSNR / SSIM against the reference (inside the anatomy), as stored by the dump.

In [ ]:
extra = [("Zero-filled", "zf"), ("Guide (PD)", "guide")]
extra = [(lab, k) for lab, k in extra if rows[0][k] is not None]
ncol = len(extra) + len(shown) + 1
r0, c0 = crop_box(rows[0]["mask"])
h, w = rows[0]["ref"][r0, c0].shape
fig, axes = plt.subplots(len(rows), ncol, figsize=(1.9 * ncol, 1.9 * h / w * len(rows) + 0.6),
                         squeeze=False)
for i, row in enumerate(rows):
    rs, cs = crop_box(row["mask"])
    vmax = window(row["ref"], row["mask"])
    show = lambda ax, a: ax.imshow(np.clip(a[rs, cs] / vmax, 0, 1) ** 0.85, cmap="gray",  # noqa: E731
                                   vmin=0, vmax=1, interpolation="nearest")
    panels = ([(lab, row[k], None) for lab, k in extra]
              + [(lab, img, m) for lab, img, m, _ in row["cols"]]
              + [("Ground truth", row["ref"], None)])
    for j, (lab, img, m) in enumerate(panels):
        ax = axes[i, j]
        show(ax, img)
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(lab, fontsize=9)
        if j == 0:
            ax.set_ylabel(row["name"], fontsize=7)
        if m:
            ax.set_xlabel(f"{m.get('psnr', np.nan):.2f} dB / {m.get('ssim', np.nan):.3f}",
                          fontsize=7)
plt.tight_layout()
plt.show()

## Residuals
|reference - method|, x `RESID_GAIN`, on the same per-row window (so a residual that fills the colour range is 1/RESID_GAIN of the image's brightness). Under each: NRMSE.

In [ ]:
rcols = ([("Zero-filled", "zf")] if rows[0]["zf"] is not None else [])
ncol = len(rcols) + len(shown)
fig, axes = plt.subplots(len(rows), ncol, figsize=(1.9 * ncol, 1.9 * h / w * len(rows) + 0.6),
                         squeeze=False)
for i, row in enumerate(rows):
    rs, cs = crop_box(row["mask"])
    vmax = window(row["ref"], row["mask"])
    panels = ([(lab, row[k], None) for lab, k in rcols]
              + [(lab, img, m) for lab, img, m, _ in row["cols"]])
    for j, (lab, img, m) in enumerate(panels):
        ax = axes[i, j]
        res = np.abs(row["ref"] - img)[rs, cs] * RESID_GAIN / vmax
        ax.imshow(np.clip(res, 0, 1), cmap="inferno", vmin=0, vmax=1, interpolation="nearest")
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f"{lab}", fontsize=9)
        if j == 0:
            ax.set_ylabel(row["name"], fontsize=7)
        if m:
            ax.set_xlabel(f"NRMSE {m.get('nrmse', np.nan):.4f}", fontsize=7)
fig.suptitle(f"|reference - recon| x {RESID_GAIN:g}", fontsize=10)
plt.tight_layout()
plt.show()

## Numbers
Mean stored metrics over the shown slices, and over every slice each run's full dump holds (dumped runs only; computed-here runs have just the shown slices).

In [ ]:
def mean_metrics(run, ds, pairs_):
    vals = {"psnr": [], "ssim": [], "nrmse": []}
    for v, s in pairs_:
        hit = find(run, v, s)
        if hit is None:
            continue
        for k, x in read(run, v, s, ds)["metrics"].items():
            vals[k].append(x)
    return {k: (np.mean(x) if x else np.nan) for k, x in vals.items()}, len(vals["psnr"])


full = set.intersection(*(slice_pairs(r) for r in dumped)) if dumped else set()
print(f"{'method':<22}{'shown: psnr':>12}{'ssim':>8}{'nrmse':>9}   "
      f"{'all dumped: psnr':>17}{'ssim':>8}{'n':>5}")
for lab, r, ds in shown:
    a, _ = mean_metrics(r, ds, SAMPLES)
    b, n = mean_metrics(r, ds, sorted(full)) if r in dumped else ({}, 0)
    print(f"{lab:<22}{a['psnr']:12.2f}{a['ssim']:8.4f}{a['nrmse']:9.4f}   "
          + (f"{b['psnr']:17.2f}{b['ssim']:8.4f}{n:5d}" if n else f"{'-':>17}"))